In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

PATCH_DIR        = Path("../outputs/patch")
CODE_CHANGES_DIR = Path("../outputs/code_changes")

patch_files        = sorted(PATCH_DIR.glob("*.json"))
code_changes_files = sorted(CODE_CHANGES_DIR.glob("*.json"))

records = [json.loads(p.read_text()) for p in code_changes_files]

print(f"Patch files (input):         {len(patch_files)}")
print(f"Code-changes files (output): {len(code_changes_files)}")

In [ ]:
patch_ids        = {p.stem for p in patch_files}
code_changes_ids = {p.stem for p in code_changes_files}

produced  = len(patch_ids & code_changes_ids)
no_output = len(patch_ids - code_changes_ids)
total_in  = len(patch_ids)

print(f"Patch CVEs fed in:                       {total_in}")
print(f"  Produced a code-changes file:          {produced}  ({produced/total_in:.1%})")
print(f"  No output (no resolvable GitHub link): {no_output}  ({no_output/total_in:.1%})")

In [ ]:
error_records = [r for r in records if r.get("error") is True]
ok_records    = [r for r in records if r.get("error") is not True]

print(f"Total output records:  {len(records)}")
print(f"  Error records:       {len(error_records)}  ({len(error_records)/len(records):.1%})")
print(f"  Valid records:       {len(ok_records)}  ({len(ok_records)/len(records):.1%})")

if error_records:
    print("\nSample error CVEs:")
    for r in error_records[:5]:
        tb = r.get("traceback", "").strip().splitlines()
        last_line = tb[-1] if tb else "(no traceback)"
        print(f"  {r.get('cve_id', '?'):20}  {last_line}")

In [ ]:
total_patches = 0
null_patches  = 0
cves_all_null  = 0
cves_some_null = 0
cves_no_null   = 0

for r in ok_records:
    patches = r.get("patches", [])
    nulls   = sum(1 for p in patches if p.get("diff") is None)
    total_patches += len(patches)
    null_patches  += nulls
    if nulls == len(patches): cves_all_null += 1
    elif nulls > 0:           cves_some_null += 1
    else:                     cves_no_null += 1

print("=== Patch-level ===")
print(f"  Total patches:          {total_patches}")
print(f"  Patches with diff:      {total_patches - null_patches}  ({(total_patches - null_patches)/total_patches:.1%})")
print(f"  Patches with null diff: {null_patches}  ({null_patches/total_patches:.1%})")
print()
print("=== CVE-level (valid records only) ===")
print(f"  All patches have diffs:       {cves_no_null}  ({cves_no_null/len(ok_records):.1%})")
print(f"  Some patches null, some not:  {cves_some_null}  ({cves_some_null/len(ok_records):.1%})")
print(f"  All patches null:             {cves_all_null}  ({cves_all_null/len(ok_records):.1%})")
print()

# Cross-reference with when.ipynb
import json as _json
from pathlib import Path as _Path
WHEN_DIR = _Path("../outputs/when")
no_date_in_when = {
    p.stem for p in WHEN_DIR.glob("*.json")
    if (r := _json.loads(p.read_text())) and
       (r.get("error") or r.get("delta_days") is None)
}
no_diff_in_cc = {r["cve_id"] for r in ok_records if all(p.get("diff") is None for p in r.get("patches", []))}
print("=== Cross-reference: null-diff here vs no-date in when.ipynb ===")
print(f"  when: {len(no_date_in_when)} CVEs with no patch date | code_changes: {len(no_diff_in_cc)} CVEs with no diff")
print(f"  Both failed (no date AND no diff):              {len(no_date_in_when & no_diff_in_cc)}")
print(f"  No date in when BUT has diff here (tag API gap): {len(no_date_in_when - no_diff_in_cc)}")
print(f"  Has date in when BUT no diff here:               {len(no_diff_in_cc - no_date_in_when)}")

In [ ]:
_COMMIT_RE = re.compile(r"github\.com/[^/]+/[^/]+/commit/[0-9a-f]{5,}")
_PR_RE     = re.compile(r"github\.com/[^/]+/[^/]+/pull/\d+")
_TAG_RE    = re.compile(r"github\.com/[^/]+/[^/]+/releases/tag/[^/?#]+")

def url_type(url: str) -> str:
    if _COMMIT_RE.search(url): return "commit"
    if _PR_RE.search(url):     return "pull_request"
    if _TAG_RE.search(url):    return "release_tag"
    return "other"

type_total = Counter()
type_null  = Counter()
for r in ok_records:
    for p in r.get("patches", []):
        t = url_type(p.get("url", ""))
        type_total[t] += 1
        if p.get("diff") is None: type_null[t] += 1

print(f"{'URL type':<15} {'Total':>7}  {'Null':>6}  {'Null %':>7}")
print("-" * 42)
for t in ["commit", "pull_request", "release_tag", "other"]:
    tot  = type_total[t]
    null = type_null[t]
    pct  = f"{null/tot:.1%}" if tot else "—"
    print(f"{t:<15} {tot:>7}  {null:>6}  {pct:>7}")

In [ ]:
_TEST_RE = re.compile(
    r"(?:^|/)tests?(?:/|$)"
    r"|(?:^|/)test_"
    r"|_test\.[a-z0-9]+$"
    r"|\.test\.[a-z0-9]+$",
    re.IGNORECASE,
)

def has_test_path(files: list) -> bool:
    return any(_TEST_RE.search(f) for f in files)

def all_diff_files(r: dict) -> list:
    return [f for p in r.get("patches", []) if p.get("diff") for f in p.get("files", [])]

cves_with_diff     = [r for r in ok_records if any(p.get("diff") for p in r.get("patches", []))]
cves_with_tests    = [r for r in cves_with_diff if has_test_path(all_diff_files(r))]
cves_without_tests = [r for r in cves_with_diff if not has_test_path(all_diff_files(r))]

n = len(cves_with_diff)
print(f"CVEs with at least one non-null diff: {n}")
print()
print(f"  Patch touches >= 1 test file:  {len(cves_with_tests)}  ({len(cves_with_tests)/n:.1%})")
print(f"  Patch has no test file at all: {len(cves_without_tests)}  ({len(cves_without_tests)/n:.1%})")

In [ ]:
patches_with_diff = [
    (r["cve_id"], p)
    for r in ok_records
    for p in r.get("patches", [])
    if p.get("diff")
]

p_with_test = [(cve, p) for cve, p in patches_with_diff if has_test_path(p.get("files", []))]
p_no_test   = [(cve, p) for cve, p in patches_with_diff if not has_test_path(p.get("files", []))]

n_p = len(patches_with_diff)
print(f"Patches with a non-null diff:  {n_p}")
print(f"  Touches test file(s): {len(p_with_test)}  ({len(p_with_test)/n_p:.1%})")
print(f"  No test file:         {len(p_no_test)}  ({len(p_no_test)/n_p:.1%})")

In [ ]:
ext_counter = Counter()
for _, p in p_with_test:
    for f in p.get("files", []):
        if _TEST_RE.search(f):
            ext = Path(f).suffix.lower() or "(no ext)"
            ext_counter[ext] += 1

print("Top 15 test file extensions:")
for ext, cnt in ext_counter.most_common(15):
    print(f"  {ext:<15}  {cnt}")